## 0. 모듈 Import

In [3]:
from pathlib import Path
import sys
import os
import json
import pickle
from itertools import chain
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PARQUET_SHARDS_DIR = PROJECT_ROOT / "data" / "interim" / "normalized_paragraphs" / "notebook_all_shards" / "shards"
HOVER_PATH = PROJECT_ROOT / "data" / "raw" / "hover" / "hover_train_release_v1.1.json"
RAW_WIKI_DIR = PROJECT_ROOT / "data" / "enwiki-20171001-pages-meta-current-withlinks-processed"
OUTPUT_PATH = PROJECT_ROOT / "data" / "interim" / "gold_mapping" / "hover_train_mapped_from_parquet.json"

from src.data.hover_loader import load_hover_json
from src.data.gold_mapper import (
    build_title_to_chunks,
    build_title_to_sentence_store,
    enrich_hover_claim_row,
)

## 1. parquet → chunks 변환

In [ ]:
PARQUET_COLUMNS = [
    "paragraph_uid",
    "paragraph_text",
    "title",
    "title_norm",
    "paragraph_id",
    "paragraph_index",
    "doc_id",
    "source_path",
    "record_id",
]

CHUNK_CHECKPOINT_DIR = PROJECT_ROOT / "data" / "interim" / "gold_mapping" / "parquet_chunk_checkpoints"
CHUNK_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def _checkpoint_path(parquet_file: str) -> Path:
    p = Path(parquet_file)
    return CHUNK_CHECKPOINT_DIR / f"{p.parent.name}_{p.stem}.pkl"

def _load_checkpoint(checkpoint_path: Path) -> list[dict] | None:
    if not checkpoint_path.exists():
        return None

    try:
        with checkpoint_path.open("rb") as f:
            return pickle.load(f)
    except Exception:
        # Corrupted checkpoint: remove and rebuild from source parquet.
        checkpoint_path.unlink(missing_ok=True)
        return None

def _save_checkpoint(checkpoint_path: Path, chunks: list[dict]) -> None:
    tmp_path = checkpoint_path.with_suffix(".tmp")
    with tmp_path.open("wb") as f:
        pickle.dump(chunks, f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp_path.replace(checkpoint_path)

def _parquet_to_chunks(parquet_file: str) -> list[dict]:
    checkpoint_path = _checkpoint_path(parquet_file)
    cached = _load_checkpoint(checkpoint_path)
    if cached is not None:
        return cached

    df = pd.read_parquet(parquet_file, columns=PARQUET_COLUMNS)
    rows = df[PARQUET_COLUMNS].itertuples(index=False, name=None)

    chunks = [
        {
            "id": paragraph_uid,
            "text": paragraph_text,
            "metadata": {
                "title": title,
                "title_norm": title_norm,
                "paragraph_id": paragraph_id,
                "paragraph_index": paragraph_index,
                "doc_id": doc_id,
                "source_path": source_path,
                "record_id": record_id,
            },
        }
        for (
            paragraph_uid,
            paragraph_text,
            title,
            title_norm,
            paragraph_id,
            paragraph_index,
            doc_id,
            source_path,
            record_id,
        ) in rows
    ]

    _save_checkpoint(checkpoint_path, chunks)
    return chunks

parquet_files = sorted(PARQUET_SHARDS_DIR.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No parquet shards found in: {PARQUET_SHARDS_DIR}")

cached_shards = sum(1 for p in parquet_files if _checkpoint_path(str(p)).exists())
print("num parquet shards:", len(parquet_files))
print("cached shards:", cached_shards)

# Threading is more stable than multiprocessing on Windows/Jupyter
NUM_WORKERS = min(12, max(4, (os.cpu_count() or 8)))
CHUNKSIZE = 32

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    chunk_lists = list(
        tqdm(
            executor.map(_parquet_to_chunks, map(str, parquet_files), chunksize=CHUNKSIZE),
            total=len(parquet_files),
            desc=f"parquet->chunks (resume) threads ({NUM_WORKERS})",
        )
    )

all_chunks = list(chain.from_iterable(chunk_lists))

print("num all_chunks:", len(all_chunks))
print(all_chunks[0])


num parquet shards: 15517
cached shards: 1698


parquet->chunks (resume) threads (8):  20%|██        | 3150/15517 [01:29<09:02, 22.81it/s] 

## 2. title lookup 생성

In [ ]:
title_to_chunks = build_title_to_chunks(all_chunks)

print("num titles in parquet:", len(title_to_chunks))
print(list(title_to_chunks.keys())[:10])
print(list(title_to_chunks.keys())[-10:])

## 3. hover train 로드 + target title 수집

In [27]:
hover_data = load_hover_json(str(HOVER_PATH))

target_titles = set()
for row in hover_data:
    for title, _sent_idx in row.get("supporting_facts", []):
        if title in title_to_chunks:
            target_titles.add(title)

print("num hover rows:", len(hover_data))
print("num target_titles:", len(target_titles))
print(list(target_titles)[:10])

num hover rows: 18171
num target_titles: 316
['Kansas City Royals', 'Edsger W. Dijkstra', 'Bank of England', 'Republic of Ireland', 'Marlon Brando', 'Carson City, Nevada', 'A. E. Housman', 'ABBA', 'Damon Runyon', "Hudson's Bay Company"]


## 4. raw wiki에서 sentence store 구축

In [28]:
SENTENCE_STORE_CHECKPOINT_PATH = PROJECT_ROOT / "data" / "interim" / "gold_mapping" / "sentence_store_checkpoint.json"

title_to_sentence_store = build_title_to_sentence_store(
    raw_wiki_dir=RAW_WIKI_DIR,
    target_titles=target_titles,
    checkpoint_path=SENTENCE_STORE_CHECKPOINT_PATH,
    checkpoint_every_n_files=25,
)

print("num sentence_store titles:", len(title_to_sentence_store))
sample_title = next(iter(title_to_sentence_store.keys()))
print("sample title:", sample_title)
print("num sentences:", len(title_to_sentence_store[sample_title]["sentence_texts"]))
print("first 3 sentences:", title_to_sentence_store[sample_title]["sentence_texts"][:3])

num sentence_store titles: 316
sample title: Aristotle
num sentences: 447
first 3 sentences: ['Aristotle', 'Aristotle ( ; <a href="Greek%20language">Greek</a>: Ἀριστοτέλης , , "Aristotélēs"; 384–322\xa0BC) was an <a href="Ancient%20Greece">ancient Greek</a> <a href="philosopher">philosopher</a> and <a href="scientist">scientist</a> born in the city of <a href="Stagira%20%28ancient%20city%29">Stagira</a>, <a href="Chalkidiki">Chalkidice</a>, on the northern periphery of <a href="Classical%20Greece">Classical Greece</a>.', 'His father, <a href="Nicomachus%20%28father%20of%20Aristotle%29">Nicomachus</a>, died when Aristotle was a child, whereafter <a href="Proxenus%20of%20Atarneus">Proxenus of Atarneus</a> became his guardian.']


## 5. sentence-level enrich

In [29]:
mapped_rows = []
unmapped_count = 0

for row in hover_data:
    supporting_facts = row.get("supporting_facts", [])
    if not supporting_facts:
        unmapped_count += 1
        continue

    enriched = enrich_hover_claim_row(
        claim_row=row,
        title_to_chunks=title_to_chunks,
        title_to_sentence_store=title_to_sentence_store,
    )

    if enriched["gold_doc_titles"] and enriched["gold_chunk_ids"]:
        mapped_rows.append(enriched)
    else:
        unmapped_count += 1

print("mapped_rows   :", len(mapped_rows))
print("unmapped_count:", unmapped_count)

mapped_rows   : 558
unmapped_count: 17613


## 6. 샘플 확인

In [30]:
for i in range(min(3, len(mapped_rows))):
    print("=" * 120)
    print("uid:", mapped_rows[i]["uid"])
    print("claim:", mapped_rows[i]["claim"])
    print("supporting_facts:", mapped_rows[i]["supporting_facts"])
    print("gold_doc_titles:", mapped_rows[i]["gold_doc_titles"])
    print("gold_chunk_ids:", mapped_rows[i]["gold_chunk_ids"])
    print("gold_alignment_status:", mapped_rows[i]["gold_alignment_status"])

uid: db3453c0-1596-40c5-8da1-7800041b393f
claim: A Pottawatomie massacre event known was Bleeding Kansas was largely brought about by the Missouri Compromise and the act which created the terrirtories of Kansas and Nebraska and was combated by the Republican Party.
supporting_facts: [['History of the United States Republican Party', 0], ['History of the United States Republican Party', 2], ['Kansas–Nebraska Act', 1], ['Pottawatomie massacre', 3]]
gold_doc_titles: ['Kansas–Nebraska Act']
gold_chunk_ids: ['kansas-nebraska act::p0000']
gold_alignment_status: [{'title': 'History of the United States Republican Party', 'sentence_idx': 0, 'status': 'title_not_in_parquet'}, {'title': 'History of the United States Republican Party', 'sentence_idx': 2, 'status': 'title_not_in_parquet'}, {'title': 'Kansas–Nebraska Act', 'sentence_idx': 1, 'status': 'normalized_match'}, {'title': 'Pottawatomie massacre', 'sentence_idx': 3, 'status': 'title_not_in_parquet'}]
uid: ccf0ad16-a9cd-47bd-a0f3-017473c33b

## 7. json 저장

In [31]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(mapped_rows, f, ensure_ascii=False, indent=2)

print("saved to:", OUTPUT_PATH)
print("num saved rows:", len(mapped_rows))

saved to: d:\AI\projects\multi-hop-rag-hover\data\interim\gold_mapping\hover_train_mapped_from_parquet.json
num saved rows: 558


## 8. 저장된 json 파일 확인

In [32]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    saved_data = json.load(f)

print("num saved rows:", len(saved_data))
print(saved_data[0].keys())
print(saved_data[0])

num saved rows: 558
dict_keys(['uid', 'claim', 'supporting_facts', 'label', 'num_hops', 'hpqa_id', 'gold_doc_titles', 'gold_chunk_ids', 'gold_alignment_status'])
{'uid': 'db3453c0-1596-40c5-8da1-7800041b393f', 'claim': 'A Pottawatomie massacre event known was Bleeding Kansas was largely brought about by the Missouri Compromise and the act which created the terrirtories of Kansas and Nebraska and was combated by the Republican Party.', 'supporting_facts': [['History of the United States Republican Party', 0], ['History of the United States Republican Party', 2], ['Kansas–Nebraska Act', 1], ['Pottawatomie massacre', 3]], 'label': 'SUPPORTED', 'num_hops': 3, 'hpqa_id': '5ab5dc44554299494045f089', 'gold_doc_titles': ['Kansas–Nebraska Act'], 'gold_chunk_ids': ['kansas-nebraska act::p0000'], 'gold_alignment_status': [{'title': 'History of the United States Republican Party', 'sentence_idx': 0, 'status': 'title_not_in_parquet'}, {'title': 'History of the United States Republican Party', 'sent